In [ ]:
# Install dependencies
!pip install statsbombpy pyarrow boto3 -q

In [ ]:
import os
import io
import warnings
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning
from google.colab import userdata

warnings.filterwarnings("ignore", category=NoAuthWarning)

In [ ]:
# Load R2 credentials from Colab secrets
# Add these in the key icon (left sidebar) before running:
#   R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_ACCOUNT_ID, R2_BUCKET
account_id = userdata.get("R2_ACCOUNT_ID")
access_key = userdata.get("R2_ACCESS_KEY_ID")
secret_key = userdata.get("R2_SECRET_ACCESS_KEY")
bucket     = userdata.get("R2_BUCKET")

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="auto"
)

# Verify connection
resp = s3.list_objects_v2(Bucket=bucket, MaxKeys=1)
print(f"Connected to R2 bucket: {bucket}")

In [ ]:
def upload_parquet(df, key):
    """Serialize a DataFrame to Parquet in memory and upload to R2."""
    buf = io.BytesIO()
    pq.write_table(pa.Table.from_pandas(df, preserve_index=False), buf)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf)


def fetch_events_for_match(match_id):
    """Fetch and clean events for a single match. Returns (match_id, df) or (match_id, None) on failure."""
    try:
        df = sb.events(match_id=match_id)
        df["match_id"] = match_id
        if "location" in df.columns:
            df["location_x"] = df["location"].apply(lambda l: l[0] if isinstance(l, list) else None)
            df["location_y"] = df["location"].apply(lambda l: l[1] if isinstance(l, list) else None)
            df = df.drop(columns=["location"])
        df = df.rename(columns={
            "type":         "type_name",
            "play_pattern": "play_pattern_name",
            "team":         "team_name",
            "player":       "player_name",
            "position":     "position_name"
        })
        return match_id, df
    except Exception as e:
        print(f"  FAILED match {match_id}: {e}")
        return match_id, None

In [ ]:
# Fetch all male competitions and their matches
comps = sb.competitions()
comps = comps[comps["competition_gender"] == "male"].reset_index(drop=True)
print(f"{len(comps)} competition-seasons to ingest")

all_matches = []
for _, row in comps.iterrows():
    try:
        matches = sb.matches(competition_id=row["competition_id"], season_id=row["season_id"])
        matches["competition_id"] = row["competition_id"]
        all_matches.append(matches)
    except Exception as e:
        print(f"  FAILED competition {row['competition_id']} season {row['season_id']}: {e}")

matches_df = pd.concat(all_matches, ignore_index=True)
print(f"{len(matches_df)} total matches")

In [ ]:
# Cast mixed-type object columns to string to avoid pyarrow type inference errors
for df in [comps, matches_df]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].where(df[col].isna(), df[col].astype(str))

# Upload competitions and matches to R2
upload_parquet(comps, "competitions/competitions.parquet")
print("competitions uploaded")

for comp_id, group in matches_df.groupby("competition_id"):
    upload_parquet(group, f"matches/competition_id={comp_id}/data_0.parquet")
print(f"matches uploaded ({matches_df["competition_id"].nunique()} partitions)")

In [ ]:
# Ingest events — one competition at a time to keep memory low
# ThreadPoolExecutor fetches matches within each competition in parallel
failed = []

for _, comp_row in comps.iterrows():
    comp_id = comp_row["competition_id"]
    season_id = comp_row["season_id"]
    comp_name = comp_row["competition_name"]

    match_ids = matches_df[
        (matches_df["competition_id"] == comp_id)
    ]["match_id"].tolist()

    if not match_ids:
        continue

    print(f"Fetching {len(match_ids)} matches — {comp_name} (season {season_id})")

    comp_events = []
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(fetch_events_for_match, mid): mid for mid in match_ids}
        for future in as_completed(futures):
            match_id, df = future.result()
            if df is not None:
                comp_events.append(df)
            else:
                failed.append(match_id)

    if not comp_events:
        print(f"  No events — skipping upload")
        continue

    combined = pd.concat(comp_events, ignore_index=True)
    combined["competition_id"] = comp_id
    upload_parquet(combined, f"events/competition_id={comp_id}/data_0.parquet")
    print(f"  uploaded {len(combined)} events")

print(f"
Done. {len(failed)} failed matches: {failed}")

In [ ]:
# Smoke test — count rows in R2 for a quick sanity check
events_count  = s3.list_objects_v2(Bucket=bucket, Prefix="events/")["KeyCount"]
matches_count = s3.list_objects_v2(Bucket=bucket, Prefix="matches/")["KeyCount"]
print(f"events partitions on R2:  {events_count}")
print(f"matches partitions on R2: {matches_count}")